# Strict 250 ms Dataset Builder

This notebook runs the strict trimmer pipeline in [research/The_real_Trimmer/build_clean_250ms_dataset.py](research/The_real_Trimmer/build_clean_250ms_dataset.py) and validates the generated dataset.

It creates:
- Class 1 gunshot clips from `Data/gun` and `Data/edge-collected-gunshot-audio`
- Class 0 non-gunshot clips from `Data/sound` (optional `Data/audio`)
- A separate noise set including uncertain clips, silence, and pitch-zero clips

All clips are written at a fixed length of 250 ms (or your configured `TARGET_MS`).

In [ ]:
%pip install -q librosa soundfile pandas numpy tqdm

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import pandas as pd
from IPython.display import display

In [ ]:
# Resolve project root that contains Data/
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'Data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'Data').exists():
    raise FileNotFoundError('Could not locate project root containing Data/.')

TRIMMER_SCRIPT = PROJECT_ROOT / 'research' / 'The_real_Trimmer' / 'build_clean_250ms_dataset.py'
if not TRIMMER_SCRIPT.exists():
    raise FileNotFoundError(f'Trimmer script not found: {TRIMMER_SCRIPT}')

print('Project root :', PROJECT_ROOT)
print('Trimmer script:', TRIMMER_SCRIPT)

In [ ]:
# Build configuration
OUTPUT_NAME = 'TRIMMED_250MS_STRICT'
TARGET_MS = 250
SAMPLE_RATE = 22050
CLASS0_RATIO = 1.0
MAX_CLASS1_PER_FILE = 1
ZERO_NOISE_COUNT = 250
INCLUDE_AUDIO_FOLDER = False
OVERWRITE = True

In [ ]:
cmd = [
    sys.executable,
    str(TRIMMER_SCRIPT),
    '--project-root', str(PROJECT_ROOT),
    '--output-name', OUTPUT_NAME,
    '--target-ms', str(TARGET_MS),
    '--sample-rate', str(SAMPLE_RATE),
    '--class0-ratio', str(CLASS0_RATIO),
    '--max-class1-per-file', str(MAX_CLASS1_PER_FILE),
    '--zero-noise-count', str(ZERO_NOISE_COUNT),
]

if INCLUDE_AUDIO_FOLDER:
    cmd.append('--include-audio-folder')
if OVERWRITE:
    cmd.append('--overwrite')

print('Running command:')
print(' '.join(cmd))

proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), text=True, capture_output=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f'Build failed with code {proc.returncode}')

OUT_ROOT = PROJECT_ROOT / 'Data' / OUTPUT_NAME
print('Output root:', OUT_ROOT)

In [ ]:
summary_path = OUT_ROOT / 'reports' / 'summary.json'
manifest_path = OUT_ROOT / 'reports' / 'manifest.csv'
rejected_path = OUT_ROOT / 'reports' / 'rejected_sources.csv'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('Summary file:', summary_path)
summary

In [ ]:
manifest_df = pd.read_csv(manifest_path)
print('Manifest rows:', len(manifest_df))
print('Split counts:')
print(manifest_df['split'].value_counts(dropna=False))

# Validate fixed clip duration
duration_ok = manifest_df['duration_ms'].dropna().eq(TARGET_MS).all()
print('All duration_ms == TARGET_MS:', bool(duration_ok))

# Validate no missing or empty output files
wav_paths = [OUT_ROOT / rel for rel in manifest_df['output_path'].dropna().tolist()]
sizes = pd.Series([p.stat().st_size if p.exists() else 0 for p in wav_paths], dtype='int64')
missing = int((sizes == 0).sum())
empty_or_header_only = int((sizes <= 44).sum())
print('Missing files:', missing)
print('Potentially empty WAV files (<= 44 bytes):', empty_or_header_only)

if missing > 0 or empty_or_header_only > 0:
    raise AssertionError('Dataset contains missing or empty clips.')

display(manifest_df.head(10))

## Manual Quality Audit

After build:
1. Spot-check random clips from `class_1_gunshot` and `class_0_nongunshot`.
2. Review `noise/uncertain_from_class0` to ensure suspicious impulses were isolated.
3. Keep `noise/pitch_zero` for ignore-class or robustness experiments.